# Treinamento Blindado (QLoRA Fine-Tuning) — Especialista Acadêmico Luiz Gabriel (Llama 3.1 8B)
**Por que esta versão é blindada contra quedas de Wi-Fi e limite de RAM:**
1. Conecta o seu **Google Drive logo na Célula 1** (você autoriza no primeiro segundo e pode ficar tranquilo).
2. Salva o seu progresso **direto no Google Drive a cada época** (`/content/drive/MyDrive/lora_luiz_treinado`).
3. Limpa a memória RAM/VRAM (`del trainer`, `gc.collect()`) antes de converter para `.gguf`, evitando que o Colab reinicie no final!

In [ ]:
# CÉLULA 1: Conectar Google Drive + Subir dataset (Salva no Drive para nunca mais precisar subir de novo!)
import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')

drive_dataset = "/content/drive/MyDrive/dataset_treino_luiz.jsonl"
if os.path.exists("dataset_treino_luiz.jsonl"):
    shutil.copy("dataset_treino_luiz.jsonl", drive_dataset)
elif not os.path.exists(drive_dataset):
    print("Selecione o arquivo dataset_treino_luiz.jsonl no seu computador:")
    uploaded = files.upload()
    shutil.copy("dataset_treino_luiz.jsonl", drive_dataset)

shutil.copy(drive_dataset, "dataset_treino_luiz.jsonl")
print("✅ Google Drive conectado e dataset garantido no Drive!")

In [ ]:
# CÉLULA 2: Instalar Unsloth
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# CÉLULA 3: Carregar Llama 3.1 8B + QLoRA e preparar os 232 exemplos
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

alpaca_prompt = """Abaixo está uma instrução que descreve uma tarefa, acompanhada de uma entrada que fornece o contexto. Escreva uma resposta que conclua adequadamente a solicitação sem qualquer metatexto.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    texts = [
        alpaca_prompt.format(inst, inp, out) + EOS_TOKEN
        for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"])
    ]
    return { "text" : texts }

dataset = load_dataset("json", data_files="dataset_treino_luiz.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Modelo e {len(dataset)} exemplos carregados!")

In [ ]:
# CÉLULA 4: Treinar salvando direto no Google Drive a cada época!
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

pasta_drive_lora = "/content/drive/MyDrive/lora_luiz_treinado"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        save_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = pasta_drive_lora,
        report_to = "none",
    ),
)

trainer_stats = trainer.train()
model.save_pretrained(pasta_drive_lora)
tokenizer.save_pretrained(pasta_drive_lora)
print(f"✅ TREINO SALVO DEFINITIVAMENTE NO GOOGLE DRIVE EM: {pasta_drive_lora}")

In [ ]:
# CÉLULA 5: Limpar memória RAM do Trainer e Exportar para GGUF (sem estourar os 12GB do Colab!)
import gc, torch, os, shutil

# Libera toda a memória ocupada pelo otimizador e dataset antes da fusão 16-bit -> GGUF
try:
    del trainer
    del dataset
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# maximum_memory_usage=0.5 faz a fusão camada por camada sem estourar os 12.7 GB de RAM do Colab gratuito
model.save_pretrained_gguf(
    "modelo_luiz_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
    maximum_memory_usage = 0.5,
)

for fname in os.listdir("modelo_luiz_gguf"):
    if fname.endswith(".gguf"):
        src = os.path.join("modelo_luiz_gguf", fname)
        dst = "/content/drive/MyDrive/luiz-academico-q4_k_m.gguf"
        shutil.copy(src, dst)
        print(f"🎉 SUCESSO TOTAL! Seu modelo GGUF está salvo no Google Drive em: {dst}")